# S3 GPU Loader Benchmark: Run:ai Model Streamer vs Vajra

Date: 15 June 2026, 19:39

Goal: benchmark `runai-model-streamer` and `vajra-streamer` against the same S3 source, on the same instance, with the same final GPU destination.

Flow:

1. Download the Hugging Face model to local disk once.
2. Upload that local model directory to S3 using `aws s3 sync`.
3. Stream the S3 safetensors files to GPU using Run:ai Model Streamer and record timing.
4. Release GPU tensors and clear CUDA cache.
5. Stream the same S3 source to GPU using Vajra and record timing.
6. Compare wall-clock load time, tensor count, and loaded tensor bytes.

Important: do not hardcode secrets in this notebook. Use environment variables for Hugging Face and AWS credentials.

In [1]:
%pip install -U \
  huggingface_hub \
  hf_transfer \
  torch \
  pandas \
  awscli \
  runai-model-streamer \
  runai-model-streamer-s3 \
  --pre vajra-streamer==0.0.46b11 --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Configuration

Set these before running upstream:

```bash
export HF_TOKEN=...
export BENCH_MODEL_ID=meta-llama/Meta-Llama-3-8B
export BENCH_S3_URI=s3://your-bucket/path/to/model
export AWS_REGION=ap-south-1
```

Optional knobs:

```bash
export BENCH_DEVICE=cuda:0
export BENCH_LOCAL_DIR=/tmp/vajra-bench-model
export RUNAI_STREAMER_CONCURRENCY=16
export VAJRA_CHUNK_WORKERS=16
export VAJRA_GPU_WORKERS=4
export VAJRA_CHUNK_SIZE_MB=128
```

In [2]:
import gc
import json
import os
import pathlib
import shutil
import subprocess
import time
from contextlib import contextmanager
from urllib.parse import urlparse

import pandas as pd
import torch
from huggingface_hub import snapshot_download

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

HF_TOKEN = os.getenv("HF_TOKEN", "<YOUR_HF_TOKEN>")
MODEL_ID = os.getenv("BENCH_MODEL_ID", "meta-llama/Meta-Llama-3-8B")
LOCAL_MODEL_DIR = pathlib.Path(os.getenv("BENCH_LOCAL_DIR", f"/tmp/vajra-bench/{MODEL_ID.replace('/', '--')}"))
S3_URI = os.getenv("BENCH_S3_URI", "s3://your-bucket/path/to/model").rstrip("/")
DEVICE = os.getenv("BENCH_DEVICE", "cuda:0")

RUNAI_CONCURRENCY = int(os.getenv("RUNAI_STREAMER_CONCURRENCY", "16"))
VAJRA_CHUNK_WORKERS = int(os.getenv("VAJRA_CHUNK_WORKERS", "128"))
VAJRA_GPU_WORKERS = int(os.getenv("VAJRA_GPU_WORKERS", "4"))
VAJRA_CHUNK_SIZE_MB = int(os.getenv("VAJRA_CHUNK_SIZE_MB", "8"))
VAJRA_LOG_LEVEL = int(os.getenv("VAJRA_LOG_LEVEL", "4"))

if not S3_URI.startswith("s3://"):
    raise ValueError("Set BENCH_S3_URI to the S3 prefix for this benchmark, e.g. s3://bucket/prefix/model")

print(json.dumps({
    "model_id": MODEL_ID,
    "local_model_dir": str(LOCAL_MODEL_DIR),
    "s3_uri": S3_URI,
    "device": DEVICE,
    "runai_concurrency": RUNAI_CONCURRENCY,
    "vajra_chunk_workers": VAJRA_CHUNK_WORKERS,
    "vajra_gpu_workers": VAJRA_GPU_WORKERS,
    "vajra_chunk_size_mb": VAJRA_CHUNK_SIZE_MB,
}, indent=2))

{
  "model_id": "meta-llama/Meta-Llama-3-8B",
  "local_model_dir": "/tmp/vajra-bench/meta-llama--Meta-Llama-3-8B",
  "s3_uri": "s3://your-bucket/path/to/model",
  "device": "cuda:0",
  "runai_concurrency": 16,
  "vajra_chunk_workers": 128,
  "vajra_gpu_workers": 4,
  "vajra_chunk_size_mb": 8
}


/home/username/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import gc
import torch

def bytes_to_gib(n: int) -> float:
    return n / (1024 ** 3)


def tensor_nbytes(tensor) -> int:
    return int(tensor.numel() * tensor.element_size())


def tensor_collection_nbytes(tensors: dict) -> int:
    return sum(tensor_nbytes(t) for t in tensors.values())


def join_s3(prefix: str, name: str) -> str:
    return prefix.rstrip("/") + "/" + name.lstrip("/")


def release_gpu_objects(*names):
    for name in names:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()


def cuda_memory_snapshot(label: str):
    if not torch.cuda.is_available():
        print(f"{label}: CUDA unavailable")
        return
    torch.cuda.synchronize()
    print(
        f"{label}: allocated={bytes_to_gib(torch.cuda.memory_allocated()):.2f} GiB, "
        f"reserved={bytes_to_gib(torch.cuda.memory_reserved()):.2f} GiB"
    )


benchmark_results = []


@contextmanager
def timed_case(name: str):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    record = {"name": name}
    try:
        yield record
    finally:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        record["seconds"] = time.perf_counter() - start
        benchmark_results.append(record)
        print(f"{name}: {record['seconds']:.3f}s")

## Step 1: Download Model To Disk Once

This prepares the benchmark source. Do not include this time in the Run:ai/Vajra comparison.

In [4]:
LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

with timed_case("hf_transfer_snapshot_download") as rec:
    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
    downloaded_path = snapshot_download(
        repo_id=MODEL_ID,
        token=HF_TOKEN or None,
        local_dir=str(LOCAL_MODEL_DIR),
        local_dir_use_symlinks=False,
        allow_patterns=[
            "*.safetensors",
            "*.safetensors.index.json",
            "config.json",
            "generation_config.json",
            "tokenizer*",
            "*.model",
            "*.json",
        ],
    )

local_model_path = pathlib.Path(downloaded_path)
local_safetensors = sorted(local_model_path.glob("*.safetensors"))
if not local_safetensors:
    raise RuntimeError(f"No .safetensors files found in {local_model_path}")

local_safetensor_bytes = sum(p.stat().st_size for p in local_safetensors)
rec["source_gib"] = bytes_to_gib(local_safetensor_bytes)
rec["tensor_count"] = 0
rec["tensor_gib"] = 0
print(f"Downloaded to: {local_model_path}")
print(f"Safetensors files: {len(local_safetensors)}")
print(f"Safetensors bytes: {bytes_to_gib(local_safetensor_bytes):.2f} GiB")
for p in local_safetensors:
    print(f"- {p.name}: {bytes_to_gib(p.stat().st_size):.2f} GiB")

/home/username/.local/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 12 files: 100%|███████████████| 12/12 [00:37<00:00,  3.09s/it]

hf_transfer_snapshot_download: 37.315s
Downloaded to: /tmp/vajra-bench/meta-llama--Meta-Llama-3-8B
Safetensors files: 4
Safetensors bytes: 14.96 GiB
- model-00001-of-00004.safetensors: 4.63 GiB
- model-00002-of-00004.safetensors: 4.66 GiB
- model-00003-of-00004.safetensors: 4.58 GiB
- model-00004-of-00004.safetensors: 1.09 GiB


## Step 2: Upload The Local Model Directory To S3

This creates the shared S3 source used by both loaders. The benchmark itself starts after this cell.

In [5]:
aws_env = os.environ.copy()
aws_env.update({
    "AWS_ACCESS_KEY_ID": "<YOUR_AWS_ACCESS_KEY_ID>",
    "AWS_SECRET_ACCESS_KEY": "<YOUR_AWS_SECRET_ACCESS_KEY>",
    "AWS_REGION": "ap-south-1",
    "AWS_DEFAULT_REGION": "ap-south-1",
})

aws_cmd = [
    "aws", "s3", "sync", str(local_model_path), S3_URI,
    "--only-show-errors",
    "--exclude", "*",
    "--include", "*.safetensors",
    "--include", "*.safetensors.index.json",
    "--include", "*.json",
    "--include", "tokenizer*",
    "--include", "*.model",
]

# aws_profile = os.getenv("AWS_PROFILE")
# if aws_profile:
#     aws_cmd.extend(["--profile", aws_profile])

# print(" ".join(aws_cmd))
# subprocess.run(aws_cmd, check=True, env=aws_env)

s3_safetensor_paths = [join_s3(S3_URI, p.name) for p in local_safetensors]
print("S3 safetensors source:")
for path in s3_safetensor_paths:
    print("-", path)

S3 safetensors source:
- s3://your-bucket/path/to/model/model-00001-of-00004.safetensors
- s3://your-bucket/path/to/model/model-00002-of-00004.safetensors
- s3://your-bucket/path/to/model/model-00003-of-00004.safetensors
- s3://your-bucket/path/to/model/model-00004-of-00004.safetensors


## Step 3: Benchmark Run:ai Model Streamer From S3 To GPU

This uses the same S3 safetensors files and attempts to materialize tensors on `BENCH_DEVICE`.

In [6]:
release_gpu_objects("runai_tensors", "vajra_tensors")
cuda_memory_snapshot("before runai")

os.environ["RUNAI_STREAMER_CONCURRENCY"] = str(RUNAI_CONCURRENCY)

from runai_model_streamer import SafetensorsStreamer

runai_tensors = {}

from runai_model_streamer.s3_utils.s3_utils import S3Credentials

os.environ.update({
    "AWS_ACCESS_KEY_ID": aws_env["AWS_ACCESS_KEY_ID"],
    "AWS_SECRET_ACCESS_KEY": aws_env["AWS_SECRET_ACCESS_KEY"],
    "AWS_REGION": "ap-south-1",
    "AWS_DEFAULT_REGION": "ap-south-1",
    "AWS_EC2_METADATA_DISABLED": "true",
    "RUNAI_STREAMER_NO_BOTO3_SESSION": "1",
    "RUNAI_STREAMER_S3_TRACE": "1",
})

s3_credentials = S3Credentials(
    access_key_id=aws_env["AWS_ACCESS_KEY_ID"],
    secret_access_key=aws_env["AWS_SECRET_ACCESS_KEY"],
    region_name="ap-south-1",
    endpoint="https://s3.ap-south-1.amazonaws.com",
)

print(s3_credentials)

with timed_case("runai_model_streamer_s3_to_gpu") as rec:
    with SafetensorsStreamer() as streamer:
        try:
            streamer.stream_files(
                s3_safetensor_paths,
                s3_credentials=s3_credentials,
                device=DEVICE,
                is_distributed=False,
            )
        except TypeError:
            # Compatibility fallback for older package signatures.
            streamer.stream_files(s3_safetensor_paths)

        for name, tensor in streamer.get_tensors():
            if DEVICE.startswith("cuda") and not tensor.is_cuda:
                tensor = tensor.to(DEVICE, non_blocking=True)
            runai_tensors[name] = tensor

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    rec["tensor_count"] = len(runai_tensors)
    rec["tensor_gib"] = bytes_to_gib(tensor_collection_nbytes(runai_tensors))
    rec["source_gib"] = bytes_to_gib(local_safetensor_bytes)

cuda_memory_snapshot("after runai")
print(f"Run:ai tensors loaded: {len(runai_tensors)}")

before runai: allocated=0.00 GiB, reserved=0.00 GiB
[environment variables redacted]
runai_model_streamer_s3_to_gpu: 15.854s
after runai: allocated=14.96 GiB, reserved=15.08 GiB
Run:ai tensors loaded: 291


## Step 4: Release Run:ai GPU Tensors

This is necessary so Vajra gets a clean GPU memory state.

In [7]:
release_gpu_objects("runai_tensors")
cuda_memory_snapshot("after runai cleanup")

after runai cleanup: allocated=0.11 GiB, reserved=0.11 GiB


## Step 5: Benchmark Vajra From The Same S3 Source To GPU

`VajraStreamer.load()` receives the S3 prefix, not the individual safetensors file list. This matches the current resolver flow.

In [8]:
import faulthandler
faulthandler.enable()

In [9]:
release_gpu_objects("vajra_tensors")
cuda_memory_snapshot("before vajra")

from vajra import StreamConfig, VajraStreamer

vajra_config = StreamConfig(
    auth_token=HF_TOKEN or "",
    chunk_size_mb=VAJRA_CHUNK_SIZE_MB,
    chunk_workers=VAJRA_CHUNK_WORKERS,
    gpu_workers=VAJRA_GPU_WORKERS,
    disable_cache=True,
    log_level=VAJRA_LOG_LEVEL,
)

with timed_case("vajra_streamer_s3_to_gpu") as rec:
    with VajraStreamer(vajra_config) as streamer:
        vajra_tensors = streamer.load(S3_URI) # streamer.load(MODEL_ID)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    rec["tensor_count"] = len(vajra_tensors)
    rec["tensor_gib"] = bytes_to_gib(tensor_collection_nbytes(vajra_tensors))
    rec["source_gib"] = bytes_to_gib(local_safetensor_bytes)

cuda_memory_snapshot("after vajra")
print(f"Vajra tensors loaded: {len(vajra_tensors)}")

before vajra: allocated=0.11 GiB, reserved=0.11 GiB
[Dlang API] stream_model entered
[Dlang API] converting url
[Dlang API] converting authToken
[Dlang API] setting defaults
[Dlang API] Preparing to spawn thread
[Dlang API] instantiating new Thread
[Dlang API] Starting thread
[Dlang API] Waiting for thread to join
[Dlang API] Thread started
[Dlang API] Setting up workers
[Dlang API] runTask has been scheduled
[Dlang API] Running event loop
[Dlang API] HTTPClientSettings configured
[Dlang API] URL starts with s3://
Fetching S3 objects from: https://your-bucket.s3.ap-south-1.amazonaws.com/?[presigned query redacted]
Phase 1: Fetching S3 ListObjects from https://your-bucket.s3.ap-south-1.amazonaws.com/?[presigned query redacted]
[Dlang API] fetchS3ListObjects: starting requestHTTP
[Dlang API] fetchS3ListObjects: setting method to GET
[Dlang API] fetchS3ListObjects: received response with statusCode: 200
[Dlang API] fetchS3ListObjects: reading body
[Dlang API] fetchS3ListObjects: body read

## Step 6: Benchmark Vajra From HF Source To GPU

`VajraStreamer.load()` receives the HF source, not the individual safetensors file list. This matches the current resolver flow.

In [10]:
release_gpu_objects("vajra_tensors")
cuda_memory_snapshot("before vajra")

from vajra import StreamConfig, VajraStreamer

vajra_config = StreamConfig(
    auth_token=HF_TOKEN or "",
    chunk_size_mb=VAJRA_CHUNK_SIZE_MB,
    chunk_workers=VAJRA_CHUNK_WORKERS,
    gpu_workers=VAJRA_GPU_WORKERS,
    disable_cache=True,
    log_level=VAJRA_LOG_LEVEL,
)

with timed_case("vajra_streamer_HF_to_gpu") as rec:
    with VajraStreamer(vajra_config) as streamer:
        vajra_tensors = streamer.load(MODEL_ID) # streamer.load(S3_URI)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    rec["tensor_count"] = len(vajra_tensors)
    rec["tensor_gib"] = bytes_to_gib(tensor_collection_nbytes(vajra_tensors))
    rec["source_gib"] = bytes_to_gib(local_safetensor_bytes)

cuda_memory_snapshot("after vajra")
print(f"Vajra tensors loaded: {len(vajra_tensors)}")

before vajra: allocated=0.11 GiB, reserved=0.11 GiB
[Dlang API] stream_model entered
[Dlang API] converting url
[Dlang API] converting authToken
[Dlang API] setting defaults
[Dlang API] Preparing to spawn thread
[Dlang API] instantiating new Thread
[Dlang API] Starting thread
[Dlang API] Waiting for thread to join
[Dlang API] Thread started
[Dlang API] Setting up workers
[Dlang API] runTask has been scheduled
[Dlang API] Running event loop
[Dlang API] HTTPClientSettings configured
[Dlang API] URL is a HuggingFace URL
Fetching model metadata from: https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B
Resolved Model ID to 4 URL(s)
[Dlang API] resolveHfUrl completed
[Dlang API] Resolved Model ID to 4 URL(s)
[Dlang API] Inside runTask: Creating DownloadManager
[Manager] Registered consumer: gpu_loader
[Dlang API] Inside runTask: DownloadManager created successfully, calling dm.call
[Manager] Queuing Target URL: https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/model-000

## Step 7: Compare Results

In [11]:
results_df = pd.DataFrame(benchmark_results)
if not results_df.empty:
    results_df["throughput_gib_per_sec"] = results_df["source_gib"] / results_df["seconds"]
    display(results_df)

    if {"runai_model_streamer_s3_to_gpu", "vajra_streamer_s3_to_gpu"}.issubset(set(results_df["name"])):
        hf_transfer_snapshot_seconds = float(results_df.loc[results_df["name"] == "hf_transfer_snapshot_download", "seconds"].iloc[-1])
        runai_seconds = float(results_df.loc[results_df["name"] == "runai_model_streamer_s3_to_gpu", "seconds"].iloc[-1])
        vajra_seconds = float(results_df.loc[results_df["name"] == "vajra_streamer_s3_to_gpu", "seconds"].iloc[-1])
        vajra_hf_seconds = float(results_df.loc[results_df["name"] == "vajra_streamer_HF_to_gpu", "seconds"].iloc[-1])

        # vajra vs runai
        vajra_runai_speedup = runai_seconds / vajra_seconds
        improvement_pct = (vajra_runai_speedup - 1.0) * 100.0
        print(f"Vajra speedup vs Run:ai: {vajra_runai_speedup:.2f}x ({improvement_pct:.1f}% faster)")

        # vajra vs hf_transfer
        vajra_HF_speedup = hf_transfer_snapshot_seconds / vajra_hf_seconds
        improvement_pct = (vajra_HF_speedup - 1.0) * 100.0
        print(f"Vajra speedup vs HF: {vajra_HF_speedup:.2f}x ({improvement_pct:.1f}% faster)")
else:
    print("No benchmark results recorded.")

,name,seconds,source_gib,tensor_count,tensor_gib,throughput_gib_per_sec
0,hf_transfer_snapshot_download,37.315396,14.957559,0,0.000000,0.400841
1,runai_model_streamer_s3_to_gpu,15.854299,14.957559,291,14.957527,0.943439
2,vajra_streamer_s3_to_gpu,12.970784,14.957559,291,14.957527,1.153173
3,vajra_streamer_HF_to_gpu,8.161826,14.957559,291,14.957527,1.832624


Vajra speedup vs Run:ai: 1.22x (22.2% faster)
Vajra speedup vs HF: 4.57x (357.2% faster)


## Cleanup

Run this when finished if you want to release GPU memory before continuing the same notebook session.

In [12]:
release_gpu_objects("runai_tensors", "vajra_tensors")
cuda_memory_snapshot("final cleanup")

final cleanup: allocated=0.11 GiB, reserved=0.11 GiB
